# Full Ensemble Training Pipeline
## Complete PPO + GRU + LightGBM Training

This notebook executes the complete ensemble training pipeline with full PPO implementation.

**Requirements:**
- Run `pip install -r requirements-training.txt` before starting
- Ensure training data is available in `data/` directory
- AWS credentials configured for S3 export (optional)

**Just run all cells to train the complete ensemble!**

In [ ]:
# Environment Setup and Validation
import os
import sys
import logging
from pathlib import Path
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("🔧 Setting up training environment...")

# Set working directory
project_root = Path("/notebooks/bot") if Path("/notebooks/bot").exists() else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📁 Working directory: {project_root}")

# Validate core dependencies
missing_deps = []
try:
    import pandas
    import numpy
    import torch
    import lightgbm
    import stable_baselines3
    import gymnasium
    import optuna
    import ta
    print("✅ All core dependencies available")
except ImportError as e:
    missing_deps.append(str(e))
    print(f"❌ Missing dependencies: {e}")
    print("💡 Run: pip install -r requirements-training.txt")

# Check GPU availability
gpu_available = torch.cuda.is_available()
if gpu_available:
    print(f"🚀 GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("🖥️ Using CPU for training")

# Check AWS credentials (optional)
aws_available = all(os.getenv(key) for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"])
if aws_available:
    print("☁️ AWS credentials available - S3 export enabled")
else:
    print("⚠️ AWS credentials missing - S3 export will be disabled")

print("\n🎯 Environment validation complete!")
if missing_deps:
    print("⚠️ Please install missing dependencies before proceeding")
else:
    print("✅ Ready for ensemble training")

In [ ]:
# Execute Full Ensemble Training
import time
from datetime import datetime

print("🚀 Starting Full Ensemble Training Pipeline")
print(f"⏰ Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

try:
    # Import and initialize the training runner
    from paperspace_mlops.paperspace_superior_training import PaperspaceTrainingRunner
    
    print("🔧 Initializing training runner...")
    runner = PaperspaceTrainingRunner()
    
    print("\n🎯 Launching full ensemble training...")
    print("Models: PPO + GRU + LightGBM")
    print("Symbols: BTCEUR, ETHEUR, ADAEUR, DOTEUR, LINKEUR")
    print("Features: 103 (PPO) / 100 (GRU/LightGBM)")
    print("Transaction Cost: 0.25%")
    print("")
    
    # Execute training with progress tracking
    start_time = time.time()
    
    result = runner.run_training(
        symbols=None,  # Use config defaults
        models=None,   # Use config defaults (PPO + GRU + LightGBM)
        quick_test=False  # Full training
    )
    
    training_duration = time.time() - start_time
    
    print("\n" + "="*60)
    print("🏆 TRAINING RESULTS")
    print("="*60)
    
    if result["status"] == "complete":
        print("✅ Training completed successfully!")
        print(f"⏱️ Duration: {training_duration/60:.1f} minutes")
        print(f"📊 Symbols trained: {len(result['symbols_trained'])}")
        print(f"🤖 Models trained: {len(result['models_trained'])}")
        print(f"📈 Symbols: {', '.join(result['symbols_trained'])}")
        print(f"🧠 Models: {', '.join(result['models_trained'])}")
        
        # S3 Export Status
        export_status = result.get('export_status', {})
        if export_status.get('export_enabled', False):
            print(f"☁️ S3 Export: ✅ {export_status.get('models_exported', 0)} models exported")
        else:
            print("☁️ S3 Export: ⚠️ Disabled (missing AWS credentials)")
            
    else:
        print("❌ Training failed or incomplete")
        print(f"Status: {result['status']}")
        if result.get('errors'):
            print(f"Errors: {result['errors']}")
    
    print(f"\n🎉 Training pipeline completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except Exception as e:
    print(f"\n💥 Training pipeline failed: {e}")
    import traceback
    traceback.print_exc()
    
    print("\n🔍 Troubleshooting tips:")
    print("• Check that data databases exist in data/ directory")
    print("• Verify all dependencies are installed")
    print("• Check available memory and disk space")
    print("• Review logs above for specific error details")

In [ ]:
# Training Results Analysis and Validation
import json
from pathlib import Path
import pandas as pd

print("📊 Analyzing training results...")

try:
    # Check for trained models
    models_dir = Path("models")
    if models_dir.exists():
        # Count models by type
        model_counts = {}
        total_models = 0
        
        for model_type in ['ppo', 'gru', 'lightgbm']:
            type_dir = models_dir / model_type
            if type_dir.exists():
                symbols = [d.name for d in type_dir.iterdir() if d.is_dir()]
                model_counts[model_type] = len(symbols)
                total_models += len(symbols)
                
                if symbols:
                    print(f"✅ {model_type.upper()}: {len(symbols)} models ({', '.join(symbols)})")
                else:
                    print(f"⚠️ {model_type.upper()}: No models found")
            else:
                print(f"❌ {model_type.upper()}: Directory not found")
                model_counts[model_type] = 0
        
        print(f"\n📈 Total models trained: {total_models}")
        
        # Check for training reports
        report_files = list(Path(".").glob("training_report_*.json"))
        if report_files:
            latest_report = max(report_files, key=lambda x: x.stat().st_mtime)
            print(f"\n📄 Latest training report: {latest_report.name}")
            
            try:
                with open(latest_report, 'r') as f:
                    report = json.load(f)
                
                summary = report.get('training_summary', {})
                if summary:
                    print(f"⏱️ Training time: {summary.get('total_training_time', 0):.1f}s")
                    print(f"📊 Avg validation score: {summary.get('average_validation_score', 0):.3f}")
                    print(f"🎯 Avg test score: {summary.get('average_test_score', 0):.3f}")
                
                model_performance = report.get('model_performance', {})
                if model_performance:
                    print("\n🏆 Model Performance Summary:")
                    for model_type, stats in model_performance.items():
                        print(f"  {model_type.upper()}: {stats.get('avg_validation_score', 0):.3f} avg score ({stats.get('count', 0)} models)")
                        
            except Exception as e:
                print(f"⚠️ Could not parse training report: {e}")
        else:
            print("⚠️ No training reports found")
            
        # Export validation
        if aws_available:
            print("\n☁️ Models ready for S3 deployment")
            print("🚀 Production servers can now import these models")
        else:
            print("\n💾 Models saved locally")
            print("💡 Configure AWS credentials to enable S3 export")
            
    else:
        print("❌ No models directory found - training may have failed")
        
except Exception as e:
    print(f"⚠️ Error analyzing results: {e}")

print("\n✅ Training analysis complete!")
print("\n🎯 Next steps:")
print("• Models are ready for production deployment")
print("• Check training reports for detailed performance metrics")
print("• Deploy to production trading servers via S3 or direct copy")